In [ ]:
# Paths (added for the public repo). Paths below are relative to this notebook;
# override with the DATA_ROOT / RUNS_ROOT environment variables.
import os
DATA_ROOT = os.environ.get('DATA_ROOT', '../../data')   # data/blouse, data/aihub_official
RUNS_ROOT = os.environ.get('RUNS_ROOT', '../../runs')   # checkpoints / metrics are written here


# 폐의류 분류 — **B4 LOCAL+IMPROVED: Grad-CAM++/Layer-CAM · 고해상도 · 다중 지표**

`baseline_B4_local` 의 로컬 구조(드라이브 불필요·split CSV 재사용·잘린 이미지 허용)를 유지하면서,
평가 로직을 업그레이드한 통합본입니다. **모델 재학습 없음** — 기존 `best.pt` 그대로 사용.

**대상 변형(4종)**: `B0` | `B1` | `B2MS` | `IMPROVED`

**평가 개선점**
1. CAM 알고리즘 선택: `gradcam` / `gradcam++` / `layercam`
2. 고해상도 CAM: `layer4`(7×7) 단독 → `layer3`(14×14)·`layer3+layer4` 결합
3. 박스 임계 스윕: 0.3 / 0.5 / 0.7 (CAM 캐시로 재계산 없이)
4. 강건 지표 추가: 기존 box-IoU·Loc@0.5·Pointing + **mask-IoU** + **Mass-in-GT**
5. `to_xyxy` 견고성 가드 복원(중첩 리스트/폴리곤 대응)

**저장 위치**: `runs/**B4_xai_improved**/<변형>/`  ← 기존 `B4_xai` 와 다른 폴더라 덮어쓰지 않음

> 사용법: `Config.MODEL_VARIANT` 만 4종 중 하나로 바꿔 4번 실행.

## 0. 환경 설치

In [ ]:
# 직접 구현 — scipy(연결요소)·seaborn 만 확인
!pip -q install scipy seaborn scikit-learn

## 1. Imports & 시드 (+ 잘린 이미지 허용)

In [ ]:
import os, json, random, time, math
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy import ndimage
from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

from sklearn.model_selection import train_test_split

# 잘린(손상) 이미지도 읽도록 허용
ImageFile.LOAD_TRUNCATED_IMAGES = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch:', torch.__version__, '| Device:', DEVICE)

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
seed_everything(42)
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

## 2. 설정 (Config)

`MODEL_VARIANT` 만 바꿔 4종을 동일 코드로 평가. CAM 기본값은 권장 조합
(**layercam + layer3+layer4**), 결과는 `B4_xai_improved` 폴더에 저장됩니다.

In [ ]:
@dataclass
class Config:
    MODEL_VARIANT: str = 'B2MS_IMP'          # 'B0' | 'B1' | 'B2MS' | 'IMPROVED'

    data_root: str = f'{DATA_ROOT}/blouse'

    ckpt_B0: str       = f'{RUNS_ROOT}/baseline_resnet50/best.pt'
    ckpt_B1: str       = f'{RUNS_ROOT}/baseline_B1_texture/best.pt'
    ckpt_B2MS: str     = f'{RUNS_ROOT}/baseline_B2_multiscale/best.pt'
    ckpt_B2MSIMP: str  = f'{RUNS_ROOT}/B2MS_improved/best.pt'
    ckpt_IMPROVED: str = f'{RUNS_ROOT}/B_improved/best.pt'

    # 기존 B4_xai 와 겹치지 않도록 다른 폴더명으로 저장
    output_dir: str = f'{RUNS_ROOT}/B4_xai_improved'

    main_classes: tuple = ('재사용', '재활용', '폐기')
    img_size: int = 224
    seed: int = 42
    val_ratio: float = 0.15
    test_ratio: float = 0.15

    texture_mid_ch: int = 32           # B1 / B2MS 재구성용
    dropout_p: float = 0.3             # IMPROVED 재구성용

    # ── XAI: 단일 데모/정성 시각화용 기본 조합 ──────────────
    cam_method: str = 'layercam'       # 'gradcam' | 'gradcam++' | 'layercam'
    cam_layers: tuple = ('layer3', 'layer4')
    cam_threshold: float = 0.5
    cam_class: str = 'gt'              # 'gt' | 'pred'

    # ── 비교 스윕 ───────────────────────────────────────────
    sweep_methods: tuple = (
        ('gradcam',   ('layer4',)),            # 기존 baseline 행
        ('gradcam++', ('layer4',)),
        ('layercam',  ('layer4',)),
        ('layercam',  ('layer3',)),
        ('layercam',  ('layer3','layer4')),
    )
    sweep_thresholds: tuple = (0.3, 0.5, 0.7)

    bbox_format: str = 'xywh'          # 'xywh' | 'xyxy'
    max_eval: int = 600                # 표본 상한(속도). None=전부
    n_viz: int = 12

cfg = Config()
ckpt_map = {'B0': cfg.ckpt_B0, 'B1': cfg.ckpt_B1,
            'B2MS': cfg.ckpt_B2MS, 'IMPROVED': cfg.ckpt_IMPROVED,
            'B2MS_IMP': cfg.ckpt_B2MSIMP}
CKPT = ckpt_map[cfg.MODEL_VARIANT]
OUT  = Path(cfg.output_dir) / cfg.MODEL_VARIANT
OUT.mkdir(parents=True, exist_ok=True)
print('Variant :', cfg.MODEL_VARIANT, '| Ckpt:', CKPT)
print('Out dir :', OUT)

## 3. 드라이브 마운트 (로컬: 스킵)

In [ ]:
print('local run: drive mount skipped')

## 4. 데이터 매니페스트

In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
STATE_MAP = {'dispose': '폐기', 'recycle': '재활용', 'reusable': '재사용'}
PREFIX_MAP = {'DP': '폐기', 'RC': '재활용', 'RU': '재사용'}

def state_from_filename(name):
    return PREFIX_MAP.get(Path(name).stem.split('_')[0].upper())

def parse_label_json(json_path, image_lookup):
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception:
        return None
    meta = data.get('meta_information', {})
    big  = meta.get('big_category', '').strip().lower()
    med  = meta.get('medium_category', 'unknown').strip().lower()
    fname = meta.get('file_name') or f'{json_path.stem}.jpg'
    main = STATE_MAP.get(big) or state_from_filename(fname)
    if main is None:
        return None
    img_path = image_lookup.get(fname) or image_lookup.get(Path(fname).stem)
    if img_path is None:
        return None
    ann = data.get('annotations', {})
    return {
        'file_path': str(img_path), 'main': main, 'sub': med,
        'pollution_bbox': [b['bbox'] for b in ann.get('pollution', [])],
        'damage_bbox':    [b['bbox'] for b in ann.get('damage', [])],
    }

def build_manifest(data_root):
    data_root = Path(data_root)
    if not data_root.exists():
        raise FileNotFoundError(f'data_root not found: {data_root}')
    image_lookup = {}
    for p in data_root.rglob('*'):
        if p.suffix.lower() in IMG_EXTS:
            image_lookup[p.name] = p
            image_lookup.setdefault(p.stem, p)
    print(f'  {len(image_lookup)} image files indexed')
    rows, skipped = [], 0
    for jp in data_root.rglob('*.json'):
        r = parse_label_json(jp, image_lookup)
        if r is None:
            skipped += 1; continue
        rows.append(r)
    df = pd.DataFrame(rows).drop_duplicates(subset='file_path').reset_index(drop=True)
    print(f'  parsed {len(df)} samples, skipped {skipped}')
    return df

df = build_manifest(cfg.data_root)
assert len(df) > 0, '매니페스트가 비었습니다. cfg.data_root 확인.'
print('오염 bbox 보유:', (df['pollution_bbox'].apply(len) > 0).sum(),
      '| 손상 bbox 보유:', (df['damage_bbox'].apply(len) > 0).sum())

## 5. Test split — B0 split CSV 재사용 (없으면 자체 분할)

In [ ]:
df = df[df['main'].isin(cfg.main_classes)].reset_index(drop=True)
main2idx = {c: i for i, c in enumerate(cfg.main_classes)}
df['main_idx'] = df['main'].map(main2idx)

SUB_CLASSES = tuple(df['sub'].value_counts().index.tolist())
sub2idx = {c: i for i, c in enumerate(SUB_CLASSES)}
df['sub_idx'] = df['sub'].map(sub2idx)

_split_dir = f'{RUNS_ROOT}/_split'
_tef = os.path.join(_split_dir, 'test_files.csv')
if os.path.exists(_tef):
    _te = set(pd.read_csv(_tef)['file_path'])
    test_df = df[df['file_path'].isin(_te)].reset_index(drop=True)
    print(f'[split: B0 CSV 재사용] Test {len(test_df)} | sub_classes={len(SUB_CLASSES)}')
else:
    print('[split: B0 CSV 없음 -> 자체 분할. B0 먼저 실행 권장]')
    train_df, temp_df = train_test_split(df, test_size=cfg.val_ratio + cfg.test_ratio,
        stratify=df['main_idx'], random_state=cfg.seed)
    _vr = cfg.val_ratio / (cfg.val_ratio + cfg.test_ratio)
    val_df, test_df = train_test_split(temp_df, test_size=1 - _vr,
        stratify=temp_df['main_idx'], random_state=cfg.seed)
    test_df = test_df.reset_index(drop=True)
    print(f'Test {len(test_df)} | sub_classes={len(SUB_CLASSES)}')

test_df['gt_boxes'] = test_df.apply(
    lambda r: list(r['pollution_bbox']) + list(r['damage_bbox']), axis=1)
eval_df = test_df[test_df['gt_boxes'].apply(len) > 0].reset_index(drop=True)
print(f'평가 대상(GT 보유): {len(eval_df)}')
print('클래스 분포:', dict(eval_df['main'].value_counts()))

In [ ]:
import os, pandas as pd

# ── (1) split 누수 점검 ───────────────────────────────
_split_dir = f'{RUNS_ROOT}/_split'
_tef = os.path.join(_split_dir, 'test_files.csv')
_trf = os.path.join(_split_dir, 'train_files.csv')
print('test_files.csv 존재:', os.path.exists(_tef))

test_paths = set(test_df['file_path'])
if os.path.exists(_trf):
    train_paths = set(pd.read_csv(_trf)['file_path'])
    leak = test_paths & train_paths
    print(f'train ∩ test 누수: {len(leak)}장 ', '✓ 누수 없음' if not leak else '⚠️ 누수 있음!')
else:
    print('⚠️ train_files.csv 없음 → 누수 직접확인 불가')

# ── (2) 재사용 클래스 점검 ────────────────────────────
print('\n[전체 df]  ', dict(df['main'].value_counts()))
print('[test_df]  ', dict(test_df['main'].value_counts()))
print('[eval_df]  ', dict(eval_df['main'].value_counts()))
reuse = df[df['main'] == '재사용']
n_box = (reuse.apply(lambda r: len(r['pollution_bbox']) + len(r['damage_bbox']), axis=1) > 0).sum()
print(f'재사용 {len(reuse)}장 중 오염/손상 박스 보유: {n_box}장')

## 6. 전처리 (학습/추론과 동일)

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
eval_tf = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
_mean = torch.tensor(MEAN).view(3, 1, 1)
_std  = torch.tensor(STD).view(3, 1, 1)
def denormalize(t):
    img = (t.cpu() * _std + _mean).clamp(0, 1)
    return img.permute(1, 2, 0).numpy()

## 7. bbox 도구 (견고성 가드 복원) + GT 마스크

In [ ]:
def to_xyxy(b, fmt):
    # [[x,y,w,h]] 처럼 한 겹 감싸진 경우
    while isinstance(b, list) and len(b) == 1 and isinstance(b[0], list):
        b = b[0]
    # polygon [[x1,y1],[x2,y2],...]
    if isinstance(b, list) and len(b) > 0 and isinstance(b[0], list):
        xs = [float(p[0]) for p in b]; ys = [float(p[1]) for p in b]
        return [min(xs), min(ys), max(xs), max(ys)]
    x, y, c, d = float(b[0]), float(b[1]), float(b[2]), float(b[3])
    if fmt == 'xywh':
        return [x, y, x + c, y + d]
    return [x, y, c, d]

def scale_box(box_xyxy, W, H, size=224):
    sx, sy = size / W, size / H
    x1, y1, x2, y2 = box_xyxy
    return [x1 * sx, y1 * sy, x2 * sx, y2 * sy]

def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def gt_mask_from_boxes(boxes_xyxy, size=224):
    m = np.zeros((size, size), dtype=bool)
    for (x1, y1, x2, y2) in boxes_xyxy:
        xi1, yi1 = max(0, int(round(x1))), max(0, int(round(y1)))
        xi2, yi2 = min(size, int(round(x2))), min(size, int(round(y2)))
        if xi2 > xi1 and yi2 > yi1:
            m[yi1:yi2, xi1:xi2] = True
    return m

def sanity_check_format(n=20):
    bad = checked = 0
    for _, r in eval_df.head(n).iterrows():
        W, H = Image.open(r['file_path']).size
        for b in r['gt_boxes']:
            x1, y1, x2, y2 = to_xyxy(b, cfg.bbox_format)
            checked += 1
            if x2 > W * 1.05 or y2 > H * 1.05 or x2 <= x1 or y2 <= y1:
                bad += 1
    print(f'bbox 포맷 점검: {checked}개 중 이상 {bad}개',
          '✓' if (not checked or bad/checked <= 0.2) else '⚠️ xywh↔xyxy 확인')
sanity_check_format()

## 8. 모델 재구성 & 체크포인트 로드 (B0/B1/B2MS/IMPROVED)

CAM 용으로 `layer3`·`layer4` 를 모두 노출합니다.

In [ ]:
class PlainResNet(nn.Module):
    def __init__(self, num_main=3, pretrained=False):
        super().__init__()
        self.backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        in_feat = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.head_main = nn.Linear(in_feat, num_main)
    def forward(self, x):
        return self.head_main(self.backbone(x))

class TextureAwareBlock(nn.Module):
    def __init__(self, mid_ch=32, out_ch=64):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(3, mid_ch, 3, 1, 1, bias=False), nn.BatchNorm2d(mid_ch), nn.ReLU(True),
            nn.Conv2d(mid_ch, out_ch, 3, 2, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, 2, 1, bias=False), nn.BatchNorm2d(out_ch))
    def forward(self, x):
        return self.body(x)

class MultiScaleTextureBlock(nn.Module):
    def __init__(self, mid_ch=32, out_ch=64):
        super().__init__()
        self.fine   = nn.Conv2d(3, mid_ch, 3, 1, 1, bias=False)
        self.coarse = nn.Conv2d(3, mid_ch, 3, 1, 2, dilation=2, bias=False)
        self.fuse_bn = nn.BatchNorm2d(mid_ch * 2); self.relu = nn.ReLU(True)
        self.down = nn.Sequential(
            nn.Conv2d(mid_ch * 2, out_ch, 3, 2, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, 2, 1, bias=False), nn.BatchNorm2d(out_ch))
    def forward(self, x):
        t = torch.cat([self.fine(x), self.coarse(x)], dim=1)
        t = self.relu(self.fuse_bn(t))
        return self.down(t)

class TextureResNet(nn.Module):
    """B1/B2-MS 공용: block_cls 만 바꿔 끼움."""
    def __init__(self, num_main=3, block_cls=TextureAwareBlock, texture_mid_ch=32, pretrained=False):
        super().__init__()
        self.backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        in_feat = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.texture_block = block_cls(mid_ch=texture_mid_ch, out_ch=64)
        self.texture_alpha = nn.Parameter(torch.zeros(1))
        self.head_main = nn.Linear(in_feat, num_main)
    def forward(self, x):
        bb = self.backbone
        s = bb.maxpool(bb.relu(bb.bn1(bb.conv1(x))))
        s = s + self.texture_alpha * self.texture_block(x)
        f = bb.layer4(bb.layer3(bb.layer2(bb.layer1(s))))
        f = torch.flatten(bb.avgpool(f), 1)
        return self.head_main(f)

class ImprovedResNet(nn.Module):
    def __init__(self, num_main=3, dropout_p=0.3, pretrained=False):
        super().__init__()
        self.backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        in_feat = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.head_main = nn.Sequential(nn.Dropout(dropout_p), nn.Linear(in_feat, num_main))
    def forward(self, x):
        return self.head_main(self.backbone(x))

class MSImprovedResNet(nn.Module):
    """멀티스케일 텍스처 + Dropout 헤드 (B2MS + 개선 레시피 학습본)."""
    def __init__(self, num_main=3, texture_mid_ch=32, dropout_p=0.3, pretrained=False):
        super().__init__()
        w = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        self.backbone = resnet50(weights=w)
        in_feat = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.texture_block = MultiScaleTextureBlock(mid_ch=texture_mid_ch, out_ch=64)
        self.texture_alpha = nn.Parameter(torch.zeros(1))
        self.head_main = nn.Sequential(nn.Dropout(dropout_p), nn.Linear(in_feat, num_main))
    def forward(self, x):
        bb = self.backbone
        s = bb.maxpool(bb.relu(bb.bn1(bb.conv1(x))))
        s = s + self.texture_alpha * self.texture_block(x)
        f = bb.layer4(bb.layer3(bb.layer2(bb.layer1(s))))
        f = torch.flatten(bb.avgpool(f), 1)
        return self.head_main(f)

def build_model(variant):
    if variant == 'B0':
        return PlainResNet(num_main=len(cfg.main_classes))
    if variant == 'B1':
        return TextureResNet(num_main=len(cfg.main_classes),
                             block_cls=TextureAwareBlock, texture_mid_ch=cfg.texture_mid_ch)
    if variant == 'B2MS':
        return TextureResNet(num_main=len(cfg.main_classes),
                             block_cls=MultiScaleTextureBlock, texture_mid_ch=cfg.texture_mid_ch)
    if variant == 'IMPROVED':
        return ImprovedResNet(num_main=len(cfg.main_classes), dropout_p=cfg.dropout_p)
    if variant == 'B2MS_IMP':
        return MSImprovedResNet(num_main=len(cfg.main_classes),
                                texture_mid_ch=cfg.texture_mid_ch, dropout_p=0.3)
    raise ValueError(variant)

model = build_model(cfg.MODEL_VARIANT).to(DEVICE)
assert Path(CKPT).exists(), f'checkpoint not found: {CKPT}'
ckpt = torch.load(CKPT, map_location=DEVICE)
state = ckpt.get('model', ckpt.get('state_dict', ckpt)) if isinstance(ckpt, dict) else ckpt
missing, unexpected = model.load_state_dict(state, strict=False)
model.eval()
print('Loaded:', CKPT, '| missing', len(missing), '| unexpected', len(unexpected))
if len(missing) > 5:
    print('  WARNING: missing 多 — MODEL_VARIANT 와 체크포인트 일치 확인')

TARGET_LAYERS = {'layer3': model.backbone.layer3, 'layer4': model.backbone.layer4}

## 9. CAM 엔진 — Grad-CAM / Grad-CAM++ / Layer-CAM (다중 레이어 결합)

여러 레이어를 주면 각 CAM 을 224 로 업샘플·정규화 후 평균해 고해상도 CAM 생성.

In [ ]:
class CAM:
    def __init__(self, model, layers_dict):
        self.model = model
        self.acts, self.grads, self.handles = {}, {}, []
        for name, layer in layers_dict.items():
            self.handles.append(layer.register_forward_hook(self._save_act(name)))
            self.handles.append(layer.register_full_backward_hook(self._save_grad(name)))
    def _save_act(self, name):
        def hook(m, i, o): self.acts[name] = o.detach()
        return hook
    def _save_grad(self, name):
        def hook(m, gi, go): self.grads[name] = go[0].detach()
        return hook
    def remove(self):
        for h in self.handles: h.remove()
        self.handles = []
    def _run(self, x, class_idx):
        self.model.zero_grad(set_to_none=True)
        out = self.model(x)
        logits = out[0] if isinstance(out, (tuple, list)) else out
        if class_idx is None:
            class_idx = int(logits.argmax(1))
        logits[0, class_idx].backward()
        return int(class_idx)
    def _single(self, name, method):
        A, G = self.acts[name], self.grads[name]
        if method == 'gradcam':
            w = G.mean(dim=(2, 3), keepdim=True)
            cam = F.relu((w * A).sum(1, keepdim=True))
        elif method == 'gradcam++':
            G2, G3 = G.pow(2), G.pow(3)
            denom = 2 * G2 + (A * G3).sum(dim=(2, 3), keepdim=True)
            denom = torch.where(denom != 0, denom, torch.ones_like(denom))
            alpha = G2 / denom
            w = (alpha * F.relu(G)).sum(dim=(2, 3), keepdim=True)
            cam = F.relu((w * A).sum(1, keepdim=True))
        elif method == 'layercam':
            cam = F.relu((F.relu(G) * A).sum(1, keepdim=True))
        else:
            raise ValueError(method)
        return cam
    def __call__(self, x, class_idx, method='gradcam', layers=None):
        cls = self._run(x, class_idx)
        layers = layers or list(self.acts.keys())
        cams = []
        for name in layers:
            c = self._single(name, method)
            c = F.interpolate(c, size=x.shape[-2:], mode='bilinear', align_corners=False)[0, 0]
            c = (c - c.min()) / (c.max() - c.min() + 1e-8)
            cams.append(c)
        cam = torch.stack(cams, 0).mean(0)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.cpu().numpy(), cls

def cam_to_box(cam, thr):
    mask = cam >= thr * cam.max()
    lbl, n = ndimage.label(mask)
    if n == 0:
        return None
    sizes = ndimage.sum(mask, lbl, index=range(1, n + 1))
    big = int(np.argmax(sizes)) + 1
    ys, xs = np.where(lbl == big)
    return [float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())]

# 셀 재실행 시 훅 중복 방지
try:
    cam_engine.remove()
except NameError:
    pass
cam_engine = CAM(model, TARGET_LAYERS)
print('CAM engine ready. layers =', list(TARGET_LAYERS.keys()))

## 10. 단일 이미지 데모 (기본 조합)

In [ ]:
def get_cam(row, method, layers, cam_class):
    img = Image.open(row['file_path']).convert('RGB')
    W, H = img.size
    x = eval_tf(img).unsqueeze(0).to(DEVICE)
    gt_idx = int(row['main_idx'])
    cls = gt_idx if cam_class == 'gt' else None
    cam, pred = cam_engine(x, class_idx=cls, method=method, layers=list(layers))
    gtb = [scale_box(to_xyxy(b, cfg.bbox_format), W, H) for b in row['gt_boxes']]
    return denormalize(eval_tf(img)), cam, gtb, gt_idx, pred

row = eval_df.iloc[0]
base, cam, gtb, gt_idx, pred = get_cam(row, cfg.cam_method, cfg.cam_layers, cfg.cam_class)
pbox = cam_to_box(cam, cfg.cam_threshold)
fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
ax[0].imshow(base); ax[0].set_title('input'); ax[0].axis('off')
ax[1].imshow(base); ax[1].imshow(cam, cmap='jet', alpha=0.45)
for g in gtb:
    ax[1].add_patch(patches.Rectangle((g[0], g[1]), g[2]-g[0], g[3]-g[1],
                    fill=False, edgecolor='lime', lw=2))
if pbox:
    ax[1].add_patch(patches.Rectangle((pbox[0], pbox[1]), pbox[2]-pbox[0], pbox[3]-pbox[1],
                    fill=False, edgecolor='red', lw=2, linestyle='--'))
ax[1].set_title(f'{cfg.cam_method} {cfg.cam_layers} | GT={cfg.main_classes[gt_idx]}/Pred={cfg.main_classes[pred]}')
ax[1].axis('off'); plt.tight_layout(); plt.show()

## 11. CAM 캐시 — (방법, 레이어) 조합당 1회 계산

In [ ]:
rows_eval = eval_df if cfg.max_eval is None else eval_df.iloc[:cfg.max_eval]

def compute_cams(method, layers, cam_class):
    recs = []
    t0 = time.time()
    for k, (_, row) in enumerate(rows_eval.iterrows()):
        img = Image.open(row['file_path']).convert('RGB')
        W, H = img.size
        x = eval_tf(img).unsqueeze(0).to(DEVICE)
        gt_idx = int(row['main_idx'])
        cls = gt_idx if cam_class == 'gt' else None
        cam, pred = cam_engine(x, class_idx=cls, method=method, layers=list(layers))
        gtb = [scale_box(to_xyxy(b, cfg.bbox_format), W, H) for b in row['gt_boxes']]
        recs.append(dict(main=row['main'], cam=cam.astype('float32'), gtb=gtb,
                         gmask=gt_mask_from_boxes(gtb, cfg.img_size),
                         gt_idx=gt_idx, pred=pred))
        if (k + 1) % 100 == 0:
            print(f'    {method}{layers} {k+1}/{len(rows_eval)} ({time.time()-t0:.0f}s)')
    return recs

## 12. 지표 — box-IoU / mask-IoU / Loc@0.5 / Pointing / Mass-in-GT

In [ ]:
def metrics_from_cams(recs, thr):
    out = []
    for r in recs:
        cam, gtb, gmask = r['cam'], r['gtb'], r['gmask']
        pbox = cam_to_box(cam, thr)
        iou_box = max((box_iou(pbox, g) for g in gtb), default=0.0) if pbox else 0.0
        pmask = cam >= thr * cam.max()
        inter = np.logical_and(pmask, gmask).sum()
        union = np.logical_or(pmask, gmask).sum()
        iou_mask = float(inter / union) if union > 0 else 0.0
        mass = float(cam[gmask].sum() / (cam.sum() + 1e-8)) if gmask.any() else 0.0
        py, px = np.unravel_index(int(np.argmax(cam)), cam.shape)
        hit = any(g[0] <= px <= g[2] and g[1] <= py <= g[3] for g in gtb)
        out.append(dict(main=r['main'], iou_box=float(iou_box), iou_mask=iou_mask,
                        mass=mass, hit=bool(hit), correct=bool(r['pred'] == r['gt_idx'])))
    return pd.DataFrame(out)

def summarize(d):
    return dict(n=int(len(d)),
                box_IoU=round(float(d['iou_box'].mean()), 4),
                mask_IoU=round(float(d['iou_mask'].mean()), 4),
                Loc_at_0p5=round(float((d['iou_box'] >= 0.5).mean()), 4),
                pointing=round(float(d['hit'].mean()), 4),
                mass_in_GT=round(float(d['mass'].mean()), 4))

## 13. 방법 × 임계 스윕 → 표 저장

In [ ]:
sweep_rows = []
cam_cache = {}
for method, layers in cfg.sweep_methods:
    key = (method, layers)
    print('CAM 계산:', key)
    cam_cache[key] = compute_cams(method, layers, cfg.cam_class)
    for thr in cfg.sweep_thresholds:
        s = summarize(metrics_from_cams(cam_cache[key], thr))
        s.update(dict(variant=cfg.MODEL_VARIANT, method=method,
                      layers='+'.join(layers), thr=thr))
        sweep_rows.append(s)

sweep = pd.DataFrame(sweep_rows)[
    ['variant','method','layers','thr','n','box_IoU','mask_IoU','Loc_at_0p5','pointing','mass_in_GT']]
sweep = sweep.sort_values(['pointing','mask_IoU'], ascending=False).reset_index(drop=True)
sweep.to_csv(OUT / 'xai_sweep.csv', index=False)
print('Saved →', OUT / 'xai_sweep.csv')
sweep

## 14. 최적 조합 상세 (전체/클래스별) 저장

In [ ]:
best = sweep.iloc[0]
best_key = (best['method'], tuple(best['layers'].split('+')))
best_res = metrics_from_cams(cam_cache[best_key], float(best['thr']))

detail = dict(variant=cfg.MODEL_VARIANT,
              best=dict(method=best['method'], layers=best['layers'], thr=float(best['thr'])),
              overall=summarize(best_res),
              per_class={c: summarize(best_res[best_res['main'] == c])
                         for c in cfg.main_classes if (best_res['main'] == c).any()},
              among_correct=(summarize(best_res[best_res['correct']])
                             if best_res['correct'].any() else {}))
with open(OUT / 'xai_best_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(detail, f, ensure_ascii=False, indent=2)
print(json.dumps(detail, ensure_ascii=False, indent=2))
print('Saved →', OUT / 'xai_best_metrics.json')

## 15. 정성 시각화 그리드 (최적 조합)

In [ ]:
# 한글 폰트 (로컬 Windows: Malgun Gothic 우선)
import matplotlib.font_manager as fm
_avail = {f.name for f in fm.fontManager.ttflist}
for _cand in ['Malgun Gothic', 'NanumGothic', 'AppleGothic', 'Gulim']:
    if _cand in _avail:
        plt.rcParams['font.family'] = _cand; break
plt.rcParams['axes.unicode_minus'] = False
print('font:', plt.rcParams['font.family'])

b_method = best['method']; b_layers = tuple(best['layers'].split('+')); b_thr = float(best['thr'])
order = best_res.sort_values('iou_box', ascending=False)
pick = order.head(cfg.n_viz // 2).index.tolist() + order.tail(cfg.n_viz - cfg.n_viz // 2).index.tolist()

cols = 4; rows_n = math.ceil(len(pick) / cols)
fig, axes = plt.subplots(rows_n, cols, figsize=(cols * 3.2, rows_n * 3.2))
axes = np.array(axes).reshape(-1)
for ax in axes:
    ax.axis('off')
for ax, ridx in zip(axes, pick):
    row = rows_eval.iloc[ridx]
    base, cam, gtb, gt_idx, pred = get_cam(row, b_method, b_layers, cfg.cam_class)
    pbox = cam_to_box(cam, b_thr)
    ax.imshow(base); ax.imshow(cam, cmap='jet', alpha=0.45)
    for g in gtb:
        ax.add_patch(patches.Rectangle((g[0], g[1]), g[2]-g[0], g[3]-g[1],
                     fill=False, edgecolor='lime', lw=1.8))
    if pbox:
        ax.add_patch(patches.Rectangle((pbox[0], pbox[1]), pbox[2]-pbox[0], pbox[3]-pbox[1],
                     fill=False, edgecolor='red', lw=1.8, linestyle='--'))
    ok = '✓' if pred == gt_idx else '✗'
    ax.set_title(f"IoU={best_res.loc[ridx,'iou_box']:.2f} {ok}", fontsize=9)
plt.suptitle(f'{cfg.MODEL_VARIANT} best: {b_method} [{best["layers"]}] thr={b_thr} (초록=GT, 빨강=예측)', y=1.005)
plt.tight_layout()
plt.savefig(OUT / 'gradcam_examples_best.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved →', OUT / 'gradcam_examples_best.png')

## 16. 해석 가이드 & 4종 비교

`MODEL_VARIANT` 를 `B0 → B1 → B2MS → IMPROVED` 로 바꿔 4번 실행하면
`runs/B4_xai_improved/<변형>/` 아래에 각각
`xai_sweep.csv` · `xai_best_metrics.json` · `gradcam_examples_best.png` 가 저장됩니다.

- **지표 신뢰 순서**: 작은 결함 GT 에는 box-IoU 가 불리 → **Pointing → Mass-in-GT → mask-IoU → box-IoU**.
- **방법 비교**: 스윕표 첫 행 `gradcam(layer4)` 가 기존 baseline. layercam·gradcam++·layer3 결합이 얼마나 올리는지 한눈에 비교.
- **4종 비교 표 만들기**: 각 변형 `xai_best_metrics.json` 의 `overall` 을 모아 정리.
- **한계**: GT 는 결함 위치이지 분류 근거와 100% 일치하지 않음(폐기/재활용에서 점수 높게 나오는 게 자연스러움).